## Sentiment Analysis on movie reviews
We are fine tuning the DistilBERT model to train on IMDb movie reviews to analyse the sentiments of a review on a movie. BERT uses masked language modelling for training the data in which the words are masked randomly in the sentences and model tries to predict the masked word.

#### Initializations

In [33]:
# Importing packages...
from datasets import load_dataset
from transformers import AutoModelForSequenceClassification, pipeline, TrainingArguments, Trainer, AutoTokenizer, DataCollatorWithPadding
import evaluate
import numpy as np

In [10]:
# Constants...
MODEL_NAME = "distilbert/distilbert-base-uncased"
SAVED_MODEL_DIR = "./model/imdb_bert_sentiment_analysis_model"

#### Testing the sentiment analysis on the bert model without fine tuning

In [20]:
pretrained_pipeline = pipeline("sentiment-analysis")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [19]:
pretrained_model_test_input = "This movie was so bad it was good!"
pretrained_pipeline(pretrained_model_test_input)
# The model gives negative sentiment for above which is incorrect... 
# Lets see if we can fine-tune the model on the IMDB dataset and get better results.

[{'label': 'NEGATIVE', 'score': 0.9757797718048096}]

#### Loading and analysing the dataset

In [2]:
# Loading the dataset...
imdb = load_dataset("imdb")

README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [5]:
imdb

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [8]:
imdb["train"][0] 
# text: review text sentence
# label: 0 for negative and 1 for positive review

{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

#### Pre-processing the dataset

In [ ]:
# Loading the tokenizer to tokenize the text data into integers (input_ids) present in the vocabulary for the model...
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [25]:
# Preprocessing function to tokenize the text data and return the input_ids and attention_mask for the model...
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True) # truncation is True to truncate the text to the maximum length allowed by the model

In [26]:
# Tokenizing the text data in the dataset using the preprocess_function defined above...
tokenized_imdb = imdb.map(preprocess_function, batched=True) # batched=True to process the data in batches for faster processing

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [31]:
tokenized_imdb
# text: review text sentence
# label: 0 for negative and 1 for positive review
# input_ids: tokenized integers for the review text
# attention_mask: mask to indicate which tokens should be used for attention and which should not (1 for tokens to be used and 0 for tokens to be ignored)
# token_type_ids: token type ids to indicate which tokens belong to which segment (not used in DistilBERT as it is a single segment model)

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 50000
    })
})

In [34]:
# Creating batches of data with padding of input_ids and attention_mask to the maximum length in the batch using DataCollatorWithPadding...
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

#### Loading metrics to evaluate the model

In [38]:
# Loading the accuracy metric for evaluation during training...
accuracy = evaluate.load("accuracy")
accuracy.description

'\nAccuracy is the proportion of correct predictions among the total number of cases processed. It can be computed with:\nAccuracy = (TP + TN) / (TP + TN + FP + FN)\n Where:\nTP: True positive\nTN: True negative\nFP: False positive\nFN: False negative\n'

In [45]:
# Function to compute the accuracy metric during evaluation...
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [40]:
# Initializing lables and ids for the model...
id2label = {0: "NEGATIVE", 1: "POSITIVE"}
label2id = {"NEGATIVE": 0, "POSITIVE": 1}

#### Training the model

In [48]:
# Initializing the model for sequence classification with the number of labels and the label mapping defined above...
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [55]:
# Initializing the trainging arguments...
training_args = TrainingArguments(
    output_dir=SAVED_MODEL_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=True,
)

In [51]:
# Initializing the trainer with all the data...
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_imdb["train"],
    eval_dataset=tokenized_imdb["test"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [54]:
# Traing the model on IMDb dataset...
# trainer.train() # Trained on google colab as it has GPU support for faster training....

##### Training Summary

| Metric | Value |
|--------|-------|
| Total Steps | 3,126 |
| Total Epochs | 2 |
| Training Time | 1:00:44 |
| Training Runtime (seconds) | 3,646.87 |
| Training Samples/Second | 13.71 |
| Training Steps/Second | 0.857 |

##### Loss Metrics

| Epoch | Training Loss | Validation Loss |
|-------|---------------|-----------------|
| 1 | 0.2212 | 0.1954 |
| 2 | 0.1466 | 0.2300 |
| **Overall** | **0.2055** | - |

##### Performance Metrics

| Epoch | Accuracy |
|-------|----------|
| 1 | 0.9255 (92.55%) |
| 2 | 0.9321 (93.21%) |

##### Model Checkpoint Details

- Model shards written: 2
- Shard 1 write time: 5.62s
- Shard 2 write time: 6.79s

##### Computation Details

| Metric | Value |
|--------|-------|
| Total FLOPs | 6.56e+15 |

#### Inference

In [56]:
classifier = pipeline("sentiment-analysis", model='./model/checkpoint-3126')

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [60]:
text = "This was a masterpiece. Not completely faithful to the books, but enthralling from beginning to end. Might be my favorite of the three."
classifier(text)

[{'label': 'POSITIVE', 'score': 0.9977509379386902}]